# Pre-M0.1 — Python Functions for IQ Signals

## Unit Objective

Learn how to write Python functions that operate on IQ signal arrays.
We focus on: `def`, parameters, `return`, local variables, calling functions,
arrays as function arguments, and shape/dtype inspection inside functions.

## What You Will Learn

1. How to define a function with `def` and `return`.
2. The difference between `print()` and `return`.
3. How to pass NumPy arrays as function arguments.
4. How to inspect shape and dtype inside a function.
5. How to compute IQ power using a function.

## IQ Data Used

We use synthetic IQ signals with the canonical layout:

```
X.shape == (N, 2, L)
```

- axis 0 → examples
- axis 1 → I/Q components (0 = I, 1 = Q)
- axis 2 → time samples

In [ ]:
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

# Create a small IQ dataset: 3 examples, 2 components, 100 samples
N, L = 3, 100
I = rng.standard_normal((N, L)).astype(np.float32)
Q = rng.standard_normal((N, L)).astype(np.float32)
X = np.stack([I, Q], axis=1)  # shape (3, 2, 100)

print(f"X.shape = {X.shape}")
print(f"X.dtype = {X.dtype}")

## 1. Defining a Function

A function takes input **parameters**, does processing, and **returns** a result.

```python
def function_name(parameter1, parameter2):
    # processing
    return result
```

In [ ]:
def describe_array(x):
    """Print basic info about a NumPy array."""
    print(f"Shape: {x.shape}")
    print(f"ndim: {x.ndim}")
    print(f"dtype: {x.dtype}")
    print(f"size: {x.size}")

describe_array(X)

The function `describe_array` receives one parameter `x` and prints its properties.
It does not return anything explicitly (returns `None`).

## 2. print() vs return — Critical Difference

`print()` displays text on screen. `return` sends a value back to the caller.

This distinction is fundamental.

In [ ]:
def bad_power(x):
    """Prints power but returns nothing useful."""
    I = x[0]
    Q = x[1]
    p = np.mean(I**2 + Q**2)
    print(f"Power = {p}")
    # No return statement → returns None

def good_power(x):
    """Returns power as a value."""
    I = x[0]
    Q = x[1]
    p = np.mean(I**2 + Q**2)
    return p

# Using bad_power
result_bad = bad_power(X[0])
print(f"bad_power returned: {result_bad}")

# Using good_power
result_good = good_power(X[0])
print(f"good_power returned: {result_good}")

### Exercise: print vs return

What happens if you try to do `result_bad + 1`?
Try it below and explain why.

In [ ]:
# Your code here
# result_bad + 1  # Uncomment and run

## 3. Functions That Return Values

Let's write functions that compute useful IQ properties.

In [ ]:
def number_of_samples(x):
    """Return the number of time samples in an IQ array.
    
    For x.shape == (2, L), returns L.
    For x.shape == (N, 2, L), returns L.
    """
    return x.shape[-1]

# Test on a single sample
sample = X[0]  # shape (2, 100)
print(f"Single sample: number_of_samples = {number_of_samples(sample)}")

# Test on full array
print(f"Full array: number_of_samples = {number_of_samples(X)}")

In [ ]:
def get_iq_components(x):
    """Extract I and Q components from a single IQ sample.
    
    Input: x.shape == (2, L)
    Returns: (I, Q) each with shape (L,)
    """
    I = x[0]
    Q = x[1]
    return I, Q

I_sample, Q_sample = get_iq_components(sample)
print(f"I.shape = {I_sample.shape}")
print(f"Q.shape = {Q_sample.shape}")
print(f"I[:5] = {I_sample[:5]}")
print(f"Q[:5] = {Q_sample[:5]}")

## 4. Computing Joint IQ Power

Power of a real-valued IQ signal:

$$P = \text{mean}(I^2 + Q^2)$$

In [ ]:
def joint_iq_power(x):
    """Compute mean power from I and Q components.
    
    Input: x.shape == (2, L) or x.shape == (N, 2, L)
    Returns: scalar power value
    """
    I = x[..., 0, :]  # All examples, I component, all samples
    Q = x[..., 1, :]  # All examples, Q component, all samples
    P = np.mean(I**2 + Q**2)
    return P

# Single sample power
P_single = joint_iq_power(sample)
print(f"Power of single sample: {P_single:.6f}")

# Full array power
P_all = joint_iq_power(X)
print(f"Power of all examples: {P_all:.6f}")

## 5. Complex Magnitude Power

Alternative method using complex representation:

$$z = I + jQ$$
$$P = \text{mean}(|z|^2)$$

In [ ]:
def complex_magnitude_power(x):
    """Compute power using complex magnitude.
    
    Input: x.shape == (2, L) or x.shape == (N, 2, L)
    Returns: scalar power value
    """
    I = x[..., 0, :]
    Q = x[..., 1, :]
    z = I + 1j * Q
    P = np.mean(np.abs(z)**2)
    return P

# Compare both methods
P_iq = joint_iq_power(sample)
P_cmplx = complex_magnitude_power(sample)

print(f"IQ method:      {P_iq:.6f}")
print(f"Complex method: {P_cmplx:.6f}")
print(f"Difference:     {abs(P_iq - P_cmplx):.2e}")

## 6. Student Exercises

### Exercise 1

Write a function `peak_to_average(x)` that returns the ratio of peak power to mean power.

Input: `x.shape == (2, L)`

Hint: use `np.max` and `joint_iq_power`.

In [ ]:
def peak_to_average(x):
    """Your code here."""
    pass

# Test
# print(f"Peak-to-average: {peak_to_average(sample):.4f}")

### Exercise 2

Write a function `describe_iq(x)` that prints:
- shape
- dtype
- number of samples
- power

Use the functions you already defined.

In [ ]:
def describe_iq(x):
    """Your code here."""
    pass

# Test
# describe_iq(sample)

## 7. Pass Criterion Challenge

Every notebook must pass three gates:
1. **PC-1**: Independent axis explanation
2. **PC-2**: Injected axis swap correction
3. **PC-3**: IQ power agreement

### PC-1 — Independent Axis Explanation

#### STUDENT ATTEMPT

Answer these questions in the Markdown cell below:

1. What does axis 0 represent?
2. What does axis 1 represent?
3. What does axis 2 represent?
4. Why does axis 1 have size 2?
5. Where are I and Q located?
6. Where are the temporal samples located?

#### STUDENT AXIS EXPLANATION

Write your answers here:

1. 
2. 
3. 
4. 
5. 
6. 

In [ ]:
# This variable must only be manually changed to True
# by the student/instructor after the independent explanation
# has been reviewed.

AXES_EXPLANATION_VERIFIED = False

### PC-2 — Injected Axis Swap

Starting from the correct array `X.shape == (N, 2, L)`,
we deliberately inject an axis swap error.

In [ ]:
# Inject the axis swap
X_swapped = np.transpose(X, (0, 2, 1))

print(f"Original shape:  {X.shape}")
print(f"Swapped shape:   {X_swapped.shape}")

#### STUDENT ATTEMPT

1. Inspect `X_swapped.shape` — what changed?
2. Which axis should contain I/Q components?
3. Why is the new shape incorrect?
4. Write the corrective operation to produce `X_fixed`.
5. Verify that `X_fixed` exactly recovers `X`.

In [ ]:
# Student: write the correction here
# X_fixed = ...  # your code

# Validation (uncomment after fixing)
# assert X_fixed.shape == X.shape
# assert X_fixed.dtype == X.dtype
# assert np.array_equal(X_fixed, X)

#### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

```python
X_fixed = np.transpose(X_swapped, (0, 2, 1))
```

In [ ]:
# Automated validation
X_fixed = np.transpose(X_swapped, (0, 2, 1))

axis_swap_corrected = (
    X_fixed.shape == X.shape
    and X_fixed.dtype == X.dtype
    and np.array_equal(X_fixed, X)
)

assert X_fixed.shape == X.shape
assert X_fixed.dtype == X.dtype
assert np.array_equal(X_fixed, X)

if axis_swap_corrected:
    print("AXIS SWAP CHECK: PASS")
else:
    print("AXIS SWAP CHECK: WAIT")

### PC-3 — IQ Power Agreement

Calculate power through TWO independent paths.

In [ ]:
# Method A: real components
I = X[..., 0, :]
Q = X[..., 1, :]
P_iq = np.mean(I**2 + Q**2)

# Method B: complex magnitude
z = I + 1j * Q
P_complex = np.mean(np.abs(z)**2)

# Validate
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

power_consistency = np.allclose(
    P_iq,
    P_complex,
    rtol=POWER_RTOL,
    atol=POWER_ATOL
)

assert power_consistency

print(f"POWER COMPONENT METHOD = {P_iq:.6f}")
print(f"POWER COMPLEX METHOD   = {P_complex:.6f}")
print(f"ABSOLUTE DIFFERENCE    = {abs(P_iq - P_complex):.2e}")
print("POWER CHECK: PASS")

## 8. PASS CRITERION GATE

In [ ]:
automatic_checks = {
    "axis_swap_corrected": axis_swap_corrected,
    "power_consistency": power_consistency,
}

automatic_pass = all(automatic_checks.values())

final_pass = (
    automatic_pass
    and AXES_EXPLANATION_VERIFIED
)

print(f"AXES EXPLANATION: {'PASS' if AXES_EXPLANATION_VERIFIED else 'WAIT'}")
print(f"AXIS SWAP CHECK: {'PASS' if axis_swap_corrected else 'WAIT'}")
print(f"POWER CHECK: {'PASS' if power_consistency else 'WAIT'}")
print()

if final_pass:
    print("PRE-M0.1 FINAL STATUS: PASS")
else:
    print("PRE-M0.1 FINAL STATUS: WAIT")